# Week 4 — Health Disparities through Geospatial Data

* * *

<div class="alert alert-success">  
    
### Learning Objectives 
    
* Understand why health and life-outcome disparities are *spatially distributed*, and what role geographic analysis plays in surfacing structural inequity.
* Get familiar with the Opportunity Atlas and the kind of data it makes available.
* Use `geopandas` to load geographic boundaries and join them to attribute data.
* Build static choropleths with `geopandas.plot()` and an interactive map with `folium`.
* Critically read maps — identify what they make visible and what they erase.
* Recognize the Modifiable Areal Unit Problem (MAUP) and spatial autocorrelation as conceptual tools for evaluating geographic claims.
</div>

### Icons Used in This Notebook
🔔 **Question**: A quick question to help you understand what's going on.<br>
💡 **Tip**: How to do something a bit more efficiently or effectively.<br>
⚠️ **Warning:** Heads-up about tricky stuff or common mistakes.<br>
💭 **Reflection**: Reflecting on ethical implications, biases, and social impact in data science.

### Sections
1. [Framing: The Geography of Inequity](#framing)
2. [What Is the Opportunity Atlas?](#atlas)
3. [Geospatial Data 101](#geo101)
4. [Setup](#setup)
5. [Loading US State Boundaries](#load)
6. [A Teaching Dataset on State Health Disparities](#teaching)
7. [Joining Data to Geometry](#merge)
8. [Choropleth Maps](#choropleth)
9. [Interactive Maps with Folium](#folium)
10. [The MAUP Problem and Spatial Autocorrelation](#maup)
11. [Reading Maps Critically](#critical)
12. [Reflection Prompts](#reflection)

<a id='framing'></a>
# 1. Framing: The Geography of Inequity

In the United States, your zip code is a better predictor of your life expectancy than your genome is. The gap between the longest- and shortest-life-expectancy census tracts in the same city can be twenty years or more.

Monday's reading, Gil et al. (2019), *Urban Inequality and the Spatial Distribution of Health Outcomes*, makes the point with force: these patterns are not random and they are not the residue of individual bad choices. They are the cumulative geography of a century of redlining, disinvestment, environmental racism, and selective state attention.

O'Sullivan et al. (2018), *Geographies of Inequality*, gives us a parallel argument for socioeconomic disparities more broadly. 

This week we'll use Python tools to make some of these patterns visible — and we'll be very honest about what mapping erases as it reveals.

<a id='atlas'></a>
# 2. What Is the Opportunity Atlas?

The **Opportunity Atlas** (opportunityatlas.org) is a public-facing data product from the Opportunity Insights group, led by Raj Chetty, Nathan Hendren, and John Friedman. It uses anonymized federal tax records linked to Census data to estimate, for each US census tract, the average outcomes (income, college attendance, incarceration, single parenthood) for children who grew up in that tract — broken out by race, gender, and parental income.

The Atlas is striking because it shifts the unit of analysis from *people* to *places*. It lets you ask: "Two children with the same parental income — one growing up here, one growing up there — what happened to them?" The answers are not subtle.

It's a powerful tool. It is also a tool that requires care.

> **Data transparency note**: Maps of health and economic outcomes by neighborhood can:
>
> - **Make structural inequity visible** — surfacing the long shadows of redlining, disinvestment, and policy neglect. This is a real public good.
> - **Stigmatize neighborhoods** — feeding the narrative that certain places are dangerous or unhealthy, which can drive further disinvestment from those same places.
> - **Erase intra-tract heterogeneity** — every neighborhood has variation. A choropleth assigns one color per tract, flattening that variation visually.
> - **Naturalize racial categories** — "non-Hispanic Black," "Hispanic/Latino," and similar labels are administratively constructed (and have shifted over time). They are useful for tracking disparities, but they are not neutral natural kinds.
> - **Privilege measurable outcomes** — the Atlas measures income, college, incarceration. It does not measure community attachment, mutual aid networks, cultural production, or many other forms of flourishing.
>
> Hold all of these as we make maps.

<a id='geo101'></a>
# 3. Geospatial Data 101

**Geospatial data** is data that's *located in space*. The two pieces:

1. **Geometry** — a polygon, point, or line with coordinates. A census tract is a polygon. A weather station is a point. A river is a line.
2. **Attributes** — the numbers and categories attached to each geometry. Median income, life expectancy, % uninsured, etc.

A **GeoDataFrame** is a pandas DataFrame with one extra column called `geometry`.

⚠️ **Warning** about CRS (Coordinate Reference Systems): every map projection distorts something — area, shape, distance, or direction. Mercator (the projection most web maps use by default) inflates the size of high-latitude regions, which is why Greenland looks the size of Africa on web maps when in reality Africa is about 14× larger. The map is never the territory.

<a id='setup'></a>
# 4. Setup

In [ ]:
#%pip install geopandas folium mapclassify

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import folium

print("geopandas:", gpd.__version__)

<a id='load'></a>
# 5. Loading US State Boundaries

We'll use a small public GeoJSON of US state boundaries. (For real Opportunity Atlas work you'd use census tract shapefiles from the Census TIGER/Line product — orders of magnitude larger.)

In [ ]:
states_url = "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json"
states = gpd.read_file(states_url)
print(states.shape)
states.head()

In [ ]:
# Quick basemap — what does the geometry look like?
fig, ax = plt.subplots(figsize=(11, 6))
states.plot(ax=ax, edgecolor="gray", color="lightyellow")
ax.set_title("US states (loaded GeoJSON)")
ax.axis("off")
plt.tight_layout()
plt.show()

<a id='teaching'></a>
# 6. A Teaching Dataset on State Health Disparities

For today's lesson we'll build a small *illustrative* dataset inline. The numbers below are realistic in shape (drawn from CDC, Census, and Opportunity Atlas-adjacent published figures around 2018-2020) but should not be cited as authoritative — they are simplified for teaching.

💡 **For your final project**, the real Opportunity Atlas data is available at opportunityatlas.org/download/ — much richer, at the census-tract level.

In [ ]:
# Illustrative state-level dataset — a teaching simplification, not authoritative.
health = pd.DataFrame({
    "state":             ["California", "Texas", "New York", "Florida", "Mississippi",
                          "West Virginia", "Hawaii", "Minnesota", "Alabama", "Massachusetts",
                          "Louisiana", "Oregon", "Georgia", "Michigan", "Connecticut"],
    "life_expectancy":   [80.9, 78.5, 80.5, 79.6, 74.4,
                          74.4, 81.6, 80.5, 75.4, 80.7,
                          75.7, 79.6, 77.2, 78.0, 80.6],
    "median_income":     [78672, 64034, 71117, 57703, 45081,
                          48850, 83102, 74593, 51734, 84385,
                          50800, 67058, 61980, 59584, 79855],
    "percent_uninsured": [7.2, 18.4, 5.2, 13.0, 13.0,
                          7.5, 4.1, 4.5, 9.7, 3.0,
                          11.4, 7.2, 13.4, 5.7, 5.9]
})
health

<a id='merge'></a>
# 7. Joining Data to Geometry

We match the attribute table (`health`) onto the geometry table (`states`) by state name. The `name` column in `states` is the join key.

In [ ]:
states_health = states.merge(health, left_on="name", right_on="state", how="left")
states_health[["name", "life_expectancy", "median_income", "percent_uninsured"]].head(20)

💡 **Tip**: `how='left'` keeps every state's geometry, even if we don't have data for it. States with no data show `NaN`. When you map them, they'll appear in a default color (often gray) — make sure the legend explains that. Missing data is a category, not an absence.

<a id='choropleth'></a>
# 8. Choropleth Maps

A **choropleth** colors regions by a value. Lighter to darker = lower to higher.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
states_health.plot(
    column="life_expectancy",
    cmap="viridis",
    edgecolor="white", linewidth=0.5,
    legend=True, missing_kwds={"color": "lightgray", "label": "no data"},
    ax=ax)
ax.set_title("Life expectancy by US state (illustrative)")
ax.axis("off")
plt.tight_layout()
plt.show()

💭 **Reflection on map #1**: which states stand out on the low end? Which on the high end? You'll likely see Mississippi, West Virginia, Louisiana, and Alabama in the lower-life-expectancy band, and Hawaii, Massachusetts, California, Minnesota, Connecticut in the higher band. 

**What this map can't show you**: huge intra-state variation. Life expectancy in Marin County, California (≈ 84) and Lake County, California (≈ 76) differ by 8 years. The state-level color hides that. We'll talk about this in §10.

In [ ]:
# A second map — same geography, different attribute
fig, ax = plt.subplots(figsize=(12, 7))
states_health.plot(
    column="percent_uninsured",
    cmap="OrRd",
    edgecolor="white", linewidth=0.5,
    legend=True, missing_kwds={"color": "lightgray", "label": "no data"},
    ax=ax)
ax.set_title("Percent uninsured by US state (illustrative)")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Compare the two variables directly
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(states_health["percent_uninsured"], states_health["life_expectancy"], s=60, alpha=0.7)
for _, row in states_health.dropna(subset=["life_expectancy"]).iterrows():
    ax.annotate(row["state"], (row["percent_uninsured"], row["life_expectancy"]),
                fontsize=8, alpha=0.7)
ax.set_xlabel("Percent uninsured")
ax.set_ylabel("Life expectancy (years)")
ax.set_title("Uninsured rate vs life expectancy (illustrative)")
plt.tight_layout()
plt.show()

🔔 **Question**: There is *visible covariation* between the two maps — places with high uninsurance often (not always) have lower life expectancy. The scatter makes this more legible than the side-by-side maps.

Does this prove that uninsurance *causes* shorter lives? Why not? What would?

<a id='folium'></a>
# 9. Interactive Maps with Folium

`folium` builds Leaflet web maps you can pan and zoom. Useful for exploring data interactively.

In [ ]:
m = folium.Map(location=[39.0, -98.0], zoom_start=4, tiles="cartodbpositron")

folium.Choropleth(
    geo_data=states_health.__geo_interface__,
    data=states_health,
    columns=["name", "life_expectancy"],
    key_on="feature.properties.name",
    fill_color="YlGnBu",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Life expectancy (illustrative)",
    nan_fill_color="lightgray",
).add_to(m)

m   # in Jupyter, this displays the interactive map inline

💡 **Tip**: `folium` is great for quick data exploration. For polished, publication-grade maps, consider sticking with `geopandas.plot()` + matplotlib or moving to a dedicated cartography library (e.g. `cartopy`, or QGIS for richer styling).

<a id='maup'></a>
# 10. The MAUP Problem and Spatial Autocorrelation

Two concepts every honest geospatial analysis needs to wrestle with.

### MAUP: the Modifiable Areal Unit Problem

The units you aggregate to *change the conclusions you can draw*. State-level analysis hides huge intra-state differences. County-level analysis hides intra-county. Tract-level hides intra-tract. Block-group hides intra-block-group. There is no "natural" unit. Every choice is an analytic choice — and politically, an aggregation that happens to combine wealthy and poor areas can wash out the inequity you came to study.

This is not a technicality. It's a structural feature of every geographic claim. When you read a map, the first question to ask is: *what scale is this aggregated to, and what would it look like at a different scale?*

### Spatial autocorrelation: nearby places are similar

Geographer Waldo Tobler's "first law of geography" (1970): *everything is related to everything else, but near things are more related than distant things*. Our maps should not surprise us if neighboring states tend to look similar — they share economies, climates, migration patterns, policy histories.

Spatial autocorrelation is *not* a problem to be eliminated. It is *evidence* that the patterns we're seeing are produced by spatial processes — which is to say, by infrastructure, by policy, by segregation, by who got the highway and who got displaced for it.

<a id='critical'></a>
# 11. Reading Maps Critically

A short checklist for evaluating any map you encounter (or make):

1. **What's the unit of analysis?** State, county, tract, block group? What does the choice hide?
2. **What's the projection?** Is it equal-area, or does it inflate certain regions? (Mercator inflates polar regions.)
3. **What's the color scale?** Sequential (low → high)? Diverging (negative → 0 → positive)? Categorical? Does it have meaningful break points?
4. **What's the source of the data?** Who collected it, with what definitions, on what schedule, with what response rate?
5. **What's missing?** What places have `NaN` and why? 
6. **What story is the map *telling*?** Maps are persuasive precisely because they look factual. Is the story being told supported by the data, or is the cartography doing the heavy lifting?

💭 **Reflection**: Apply this checklist to a map you've encountered recently — in the news, in a paper, on social media. What does the checklist surface that you hadn't noticed?

### Other tools you could use for geospatial work

`geopandas` and `folium` are two great ways into geospatial analysis from Python — but the GIS world is much bigger than Python. We'll mention these in class:

- **QGIS** — a free, open-source desktop GIS application. Much richer cartographic styling, classification, projection handling, and spatial-analysis tools than Python libraries; standard in many research and policy workflows. **Highly recommended** if you want to go deeper into mapping for a final project.
- **ArcGIS** (Esri) — the industry-standard commercial GIS, used widely in government and consulting. Free for UC Berkeley students through the campus license.
- **R** with `sf` and `tmap` / `ggplot2` — equivalent power to `geopandas` + `matplotlib`, popular in spatial statistics and academic publishing.
- **Mapbox / Kepler.gl** — for interactive web maps with large datasets and richer styling than `folium`.
- **Datawrapper / Tableau / Flourish** — no-code map builders, great for journalism and quick reports.
- **CARTO** — cloud-based platform combining SQL, Python, and visualization.

For a final project, QGIS pairs particularly well with the analysis we did here — you can do the data prep in `geopandas`, then export and style the map in QGIS for publication-quality output.

<a id='reflection'></a>
# 12. Reflection Prompts

For your 300-word reflection on geospatial inequalities, you can start from any of:

1. Gil et al. argue that urban health disparities are *spatially produced*, not spatially incidental. Apply this argument to your home town, or a place you know well: what historical decisions (zoning, redlining, freeway routing, public-school funding) might be visible in a present-day map of health outcomes there?

2. The data transparency note in §2 argues that mapping inequity can simultaneously expose *and* stigmatize. Pick a real Opportunity-Atlas-style map (you can browse opportunityatlas.org) and write 100 words on what it makes visible and another 100 on what it might erase or harm.

3. Apply the MAUP discussion (§10) to a specific policy you care about. How does the choice of geographic unit shape what kinds of policy responses look feasible or worth doing?

4. Maps look factual. They are also rhetorical. Pick one of the maps you generated above and rewrite the title and legend to *change* the story it tells, without changing the data. What does that exercise reveal?

<div class="alert alert-success">

## ❗ Key Points

* Health and life-outcome disparities are spatially distributed — your zip code is a stronger predictor of life expectancy than your genome.
* The Opportunity Atlas is a powerful data product that maps childhood-neighborhood effects on adult outcomes; like all powerful tools, it can illuminate or stigmatize depending on how it's used.
* `geopandas` lets us treat geometry like just-another-pandas-column. Choropleths, joins, and filters all work the way they do in regular pandas.
* The MAUP problem means every geographic conclusion depends on the unit you aggregate to. There is no neutral unit.
* Spatial autocorrelation is evidence that the patterns we see are produced by spatial processes — infrastructure, policy, segregation — not by individual behavior alone.
* Maps are persuasive *because* they look factual. Read them critically: who made them, at what scale, with what categories, and what is the missing/`NaN` data telling us?

</div>